# Day 6: Variational Algorithms: VQE and QAOA

**Clemson Quantum Club** · SC Quantathon v3 Bootcamp · [clemsonquantum.com](https://clemsonquantum.com)

In this notebook:
1. Setup
2. Parameterized circuits
3. Hamiltonians, energies, and the variational principle
4. VQE: finding the ground state of a molecule
5. Gradients and optimizers: the parameter-shift rule, gradient descent, SPSA, COBYLA
6. VQE under noise: shots, gate error, and readout mitigation
7. Ansatz design and the dissociation curve of $\mathrm{H}_2$
8. QAOA: a combinatorial problem as a Hamiltonian
9. An energy on the live machine (optional)

> Cells marked **Your turn** have a few lines for you to fill in. Cells marked **Checkpoint** contain `assert` statements: if the cell runs without an error, the answer above it is correct. Solutions are in the companion solutions notebook.

## 1. Setup

Everything today is a loop: a classical optimizer proposes circuit parameters, the quantum side (here the exact simulator) returns an expectation value, and the optimizer adjusts. The pieces are Day 4's Estimator and entangling gates, Day 5's noise model, and `scipy.optimize`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

np.set_printoptions(precision=4, suppress=True)

SHOTS = 1000
RUN_ON_HARDWARE = False     # section 9 only; False uses the recorded run
rng = np.random.default_rng(seed=3)

sampler = StatevectorSampler(seed=np.random.default_rng(7))
estimator = StatevectorEstimator()

def counts_of(qc, shots=SHOTS):
    return sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()

try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(name="scqv3")
    print("IBM account 'scqv3' loaded.")
except Exception as e:
    service = None
    print("No saved IBM account:", type(e).__name__, "(fine until section 9)")

## 2. Parameterized circuits

A rotation gate takes an angle. Leave the angle as a symbol and the circuit becomes a function of it: a **parameterized circuit**. `Parameter` makes one symbol, `ParameterVector` makes many, and `assign_parameters` turns the symbolic circuit into a concrete one.

In [ ]:
theta = Parameter("theta")
qc = QuantumCircuit(1)
qc.ry(theta, 0)
print(qc.draw())
print("parameters:", list(qc.parameters))

for value in (0, np.pi / 2, np.pi):
    bound = qc.assign_parameters({theta: value})
    print(f"theta = {value:.3f}: P(1) = {Statevector(bound).probabilities()[1]:.3f}")

An **ansatz** is a parameterized circuit chosen as the family of states an algorithm will search through. The one below alternates a layer of $R_y$ rotations on every qubit with a ladder of CNOTs, `reps` times, and ends with one more rotation layer. It is called **hardware efficient** because every gate is cheap on a real chip. With $n$ qubits and `reps` entangling layers it has $n\,(\text{reps} + 1)$ parameters.

In [ ]:
def hardware_efficient(n, reps):
    params = ParameterVector("theta", n * (reps + 1))
    qc = QuantumCircuit(n)
    k = 0
    for layer in range(reps + 1):
        for q in range(n):
            qc.ry(params[k], q)
            k += 1
        if layer < reps:
            for q in range(n - 1):
                qc.cx(q, q + 1)
            qc.barrier()
    return qc

ansatz_qc = hardware_efficient(2, reps=1)
print(ansatz_qc.num_parameters, "parameters")
ansatz_qc.draw("mpl")

In [ ]:
# Checkpoint: parameter count, and assigning random angles gives a normalized state
assert ansatz_qc.num_parameters == 2 * (1 + 1)
random_angles = rng.uniform(0, 2 * np.pi, ansatz_qc.num_parameters)
sv = Statevector(ansatz_qc.assign_parameters(random_angles))
assert np.isclose(np.sum(np.abs(sv.data) ** 2), 1)
print("state for random angles:", sv.data)

## 3. Hamiltonians, energies, and the variational principle

A physical system's energy is an observable, its **Hamiltonian** $H$, written as a sum of Pauli strings with real coefficients. Its eigenvalues are the allowed energies and the smallest one, $E_0$, is the **ground-state energy**. For a two-qubit $H$ the matrix is $4\times4$ and `np.linalg.eigvalsh` gives $E_0$ exactly; the point of a quantum algorithm is that this stops being possible around 30 qubits (Day 4).

The **variational principle** says that for every state, $\langle\psi|H|\psi\rangle \ge E_0$, with equality only for a state in the ground eigenspace. So searching over an ansatz for the lowest energy can only approach $E_0$ from above, and the lowest energy found is an upper bound on the true one.

The Hamiltonian below is the hydrogen molecule $\mathrm{H}_2$ at its equilibrium bond length, 0.735 Å, reduced to two qubits by symmetry. The coefficients are the standard values generated by Qiskit Nature (STO-3G basis, parity mapping) and used throughout Qiskit's tutorials; O'Malley et al. (2016) ran the same two-qubit $\mathrm{H}_2$ problem on hardware. Its lowest eigenvalue, $-1.8573$ hartree, is the **electronic** energy; adding the nuclear repulsion of the two protons at this distance, $+0.7199$ hartree, gives the molecular ground-state energy $-1.1373$ hartree.

In [ ]:
H_mol = SparsePauliOp.from_list([
    ("II", -1.052373245772859),
    ("IZ", 0.39793742484318045),
    ("ZI", -0.39793742484318045),
    ("ZZ", -0.01128010425623538),
    ("XX", 0.18093119978423156),
])
print(H_mol)

eigenvalues = np.linalg.eigvalsh(H_mol.to_matrix())
E0 = eigenvalues.min()
print("\nenergies:", eigenvalues)
print("ground-state energy E0 =", E0, "hartree")

### Your turn: the energy of an ansatz state

Complete `energy(params)`: bind `params` to `ansatz_qc`, and return $\langle\psi(\theta)|H_{\rm mol}|\psi(\theta)\rangle$ as a float using `estimator`. An Estimator pub can carry the parameter values directly: `estimator.run([(circuit, observable, params)])`.

In [ ]:
def energy(params):
    result = None

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return result

In [ ]:
# Checkpoint: agrees with a direct state-vector calculation, and never goes below E0
test_params = rng.uniform(0, 2 * np.pi, ansatz_qc.num_parameters)
e = energy(test_params)
assert e is not None, "energy returns None: fill in the cell above"
direct = Statevector(ansatz_qc.assign_parameters(test_params)).expectation_value(H_mol).real
assert np.isclose(e, direct, atol=1e-9), (e, direct)
for _ in range(20):
    assert energy(rng.uniform(0, 2 * np.pi, ansatz_qc.num_parameters)) >= E0 - 1e-9, "variational principle violated"
print(f"energy(test_params) = {e:.4f} hartree  (E0 = {E0:.4f}); twenty random states all lie above E0")

## 4. VQE: finding the ground state

### 4.1 One parameter, by eye

The **variational quantum eigensolver** (VQE) minimizes $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$ over the ansatz parameters. With one parameter the whole landscape can be drawn. The ansatz below, $X$ on qubit 0 then $R_y(\theta)$ on qubit 1 and a CNOT, sweeps through the states $\cos\frac\theta2|01\rangle + \sin\frac\theta2|10\rangle$ (up to sign), which is exactly the family that contains the $\mathrm{H}_2$ ground state.

In [ ]:
def one_parameter(t):
    qc = QuantumCircuit(2)
    qc.x(0)
    qc.ry(t, 1)
    qc.cx(1, 0)
    return qc

ts = np.linspace(-np.pi, np.pi, 121)
landscape = [float(estimator.run([(one_parameter(t), H_mol)]).result()[0].data.evs) for t in ts]
t_best = ts[int(np.argmin(landscape))]

plt.plot(ts, landscape); plt.axhline(E0, color="0.4", ls="--", label="exact E0")
plt.plot(t_best, min(landscape), "o", label=f"minimum at theta = {t_best:.3f}")
plt.xlabel("theta"); plt.ylabel("energy (hartree)"); plt.legend(frameon=False); plt.show()

# Checkpoint: the one-parameter family reaches the ground state
assert min(landscape) - E0 < 1e-3

### 4.2 Four parameters, by optimizer

With more parameters the landscape cannot be drawn and a classical optimizer walks it instead. COBYLA is the usual choice on real hardware because it needs no gradients and tolerates noisy energies. `traced_energy` records every energy the optimizer asks for, so the run can be plotted afterwards.

### Your turn: run VQE

Call `scipy.optimize.minimize` on `traced_energy`, starting from `x0`, with `method="COBYLA"` and `options={"maxiter": 200}`, and store the return value in `result`.

In [ ]:
trace = []

def traced_energy(params):
    e = energy(params)
    trace.append(e)
    return e

x0 = rng.uniform(0, 2 * np.pi, ansatz_qc.num_parameters)
result = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint: converged to the ground-state energy
assert result is not None, "result is None: fill in the cell above"
assert abs(result.fun - E0) < 1e-2, f"VQE energy {result.fun:.4f} is not within 0.01 of E0 = {E0:.4f}"
print(f"VQE energy {result.fun:.6f}   exact {E0:.6f}   error {result.fun - E0:.2e} hartree   after {len(trace)} energy evaluations")
print("optimal angles:", result.x)

plt.plot(trace, ".-"); plt.axhline(E0, color="0.4", ls="--", label="exact E0")
plt.xlabel("energy evaluation"); plt.ylabel("energy (hartree)"); plt.legend(frameon=False); plt.title("VQE convergence, COBYLA"); plt.show()

The optimizer should reach the exact energy to a few decimal places from this start; the Checkpoint asks for $10^{-2}$, and the printout above shows how far it got and how many evaluations it took. Three things make the real problem harder. On hardware each energy is estimated from shots, so the optimizer sees noise of order $1/\sqrt N$ and gate error on top. Larger molecules need many more Pauli terms and more qubits, and the ansatz must be expressive enough to contain the ground state. And with many parameters the landscape can turn flat, so gradients vanish (a **barren plateau**) and the optimizer stalls. Ansatz design and optimizer choice are where most of the research in this area lives.

## 5. Gradients and optimizers

### 5.1 The parameter-shift rule

COBYLA never asked for a derivative. Gradient-based optimizers need $\partial E/\partial\theta_k$, and for circuits built from rotation gates there is an exact formula for it. Every gate in the ansatz is $R_y(\theta) = e^{-i\theta Y/2}$ with $Y^2 = I$, and for such a gate the energy as a function of that one angle, with all the others held fixed, is a sinusoid:

$$E(\theta_k) = a + b\cos\theta_k + c\sin\theta_k .$$

Its derivative is $-b\sin\theta_k + c\cos\theta_k$. Evaluating the sinusoid a quarter turn to either side gives $E(\theta_k \pm \frac{\pi}{2}) = a \mp b\sin\theta_k \pm c\cos\theta_k$, so

$$\frac{\partial E}{\partial\theta_k} = \frac{1}{2}\Big[E\big(\theta_k + \tfrac{\pi}{2}\big) - E\big(\theta_k - \tfrac{\pi}{2}\big)\Big].$$

This is the **parameter-shift rule**. It is exact, not an approximation, and the two points are far apart, so on hardware the shot noise is not divided by a small step $h$ the way it is in a finite difference. A full gradient of $m$ parameters costs $2m$ energy evaluations.

### Your turn: the gradient

Complete `parameter_shift_gradient(params)` so that it returns the gradient of `energy` as a NumPy array of length `len(params)`, one parameter-shift formula per entry.

In [ ]:
def parameter_shift_gradient(params):
    grad = np.zeros(len(params))

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return grad

In [ ]:
# Checkpoint: agrees with a central finite difference
assert energy(test_params) is not None, "complete energy in section 3 first"
def finite_difference_gradient(params, h=1e-4):
    grad = np.zeros(len(params))
    for k in range(len(params)):
        step = np.zeros(len(params)); step[k] = h
        grad[k] = (energy(params + step) - energy(params - step)) / (2 * h)
    return grad

g_shift = parameter_shift_gradient(test_params)
g_fd = finite_difference_gradient(test_params)
assert np.any(g_shift != 0), "gradient is all zeros: fill in the cell above"
assert np.allclose(g_shift, g_fd, atol=1e-6), (g_shift, g_fd)
print("parameter shift:  ", g_shift)
print("finite difference:", g_fd)

### 5.2 Three optimizers on the same problem

**Gradient descent** steps against the gradient, $\theta \leftarrow \theta - \eta\,\nabla E$, and costs $2m + 1$ evaluations per step. **SPSA** (simultaneous perturbation stochastic approximation) estimates the whole gradient from two evaluations no matter how many parameters there are: pick a random direction $\Delta$ with entries $\pm1$ and use

$$\hat g = \frac{E(\theta + c_k\Delta) - E(\theta - c_k\Delta)}{2c_k}\,\Delta ,\qquad \theta \leftarrow \theta - a_k\,\hat g ,$$

with gains $a_k = a/(k + 1 + A)^{0.602}$ and $c_k = c/(k+1)^{0.101}$ that shrink over the run (the exponents are Spall's recommended values). The estimate is noisy but unbiased on average, which is why SPSA is the standard choice on real hardware. **COBYLA** fits a linear model to the last few evaluations and never touches a gradient. All three start from the same `x0` as section 4.2.

In [ ]:
assert result is not None, "complete the VQE cell in section 4.2 first"
evaluations = [0]

def counted_energy(params):
    evaluations[0] += 1
    return energy(params)

def gradient_descent(f, x0, lr=1.0, steps=100):
    x = np.array(x0, dtype=float)
    history = [f(x)]
    for _ in range(steps):
        grad = np.zeros(len(x))
        for k in range(len(x)):
            step = np.zeros(len(x)); step[k] = np.pi / 2
            grad[k] = 0.5 * (f(x + step) - f(x - step))
        x = x - lr * grad
        history.append(f(x))
    return x, history

def spsa(f, x0, steps=200, a=2.0, c=0.2, A=10, seed=11):
    r = np.random.default_rng(seed)
    x = np.array(x0, dtype=float)
    history = [f(x)]
    for k in range(steps):
        a_k, c_k = a / (k + 1 + A) ** 0.602, c / (k + 1) ** 0.101
        delta = r.choice([-1, 1], size=len(x))
        g_hat = (f(x + c_k * delta) - f(x - c_k * delta)) / (2 * c_k) * delta
        x = x - a_k * g_hat
        history.append(f(x))
    return x, history

runs = {}
evaluations[0] = 0; x_gd, hist_gd = gradient_descent(counted_energy, x0); runs["gradient descent"] = (hist_gd, evaluations[0])
evaluations[0] = 0; x_sp, hist_sp = spsa(counted_energy, x0);             runs["SPSA"] = (hist_sp, evaluations[0])
evaluations[0] = 0; cobyla_trace = []
r_cob = minimize(lambda p: cobyla_trace.append(counted_energy(p)) or cobyla_trace[-1], x0, method="COBYLA", options={"maxiter": 200})
runs["COBYLA"] = (cobyla_trace, evaluations[0])

for name, (hist, n_eval) in runs.items():
    print(f"{name:<17} final energy {hist[-1]:.6f}   error {hist[-1] - E0:.1e}   {n_eval} energy evaluations")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for name, (hist, n_eval) in runs.items():
    ax[0].plot(hist, label=name)
    ax[1].semilogy(np.maximum(np.array(hist) - E0, 1e-12), label=name)
for a_ in ax: a_.set_xlabel("optimizer step"); a_.legend(frameon=False)
ax[0].axhline(E0, color="0.4", ls="--"); ax[0].set_ylabel("energy (hartree)"); ax[1].set_ylabel("energy - E0")
plt.tight_layout(); plt.show()

# Checkpoint: all three reach the ground-state energy
for name, (hist, n_eval) in runs.items():
    assert hist[-1] - E0 < 1e-2, f"{name} ended at {hist[-1]:.4f}"

Gradient descent is the most predictable and the most expensive per step. SPSA takes many small, noisy steps at two evaluations each. COBYLA is the cheapest here, on an exact energy; with shot noise its linear models are fooled more easily, which is why SPSA and the parameter-shift rule matter once the energy comes from a real device.

## 6. VQE under noise

### 6.1 What an Estimator measures

The exact Estimator read $\langle H\rangle$ off the state vector. A processor has to build it from counts. $H_{\rm mol}$ has four non-trivial terms. $IZ$, $ZI$, and $ZZ$ are diagonal in the computational basis, so one circuit measured in the $Z$ basis serves all three: $\langle Z_0\rangle = P(q_0 = 0) - P(q_0 = 1)$, likewise for $Z_1$, and $\langle Z_0Z_1\rangle = P(\text{agree}) - P(\text{differ})$. $XX$ needs a Hadamard on each qubit before measuring, as in Day 4, and then the same agree-minus-differ rule. The energy is the coefficient-weighted sum, and each term carries the shot noise of Day 4.

In [ ]:
assert result is not None, "complete the VQE cell in section 4.2 first"
theta_opt = result.x
bound = ansatz_qc.assign_parameters(theta_opt)
E_exact = energy(theta_opt)
labels = ["00", "01", "10", "11"]
coef = {label: c.real for label, c in H_mol.to_list()}

def z_circuit(qc):
    c = qc.copy(); c.measure_all(); return c

def x_circuit(qc):
    c = qc.copy(); c.h([0, 1]); c.measure_all(); return c

def probs_of(counts):
    n = sum(counts.values())
    return {k: counts.get(k, 0) / n for k in labels}

def z_terms(probs):
    z0 = sum(p * (1 if k[1] == "0" else -1) for k, p in probs.items())     # qubit 0 is the right character
    z1 = sum(p * (1 if k[0] == "0" else -1) for k, p in probs.items())
    zz = sum(p * (1 if k[0] == k[1] else -1) for k, p in probs.items())
    return z0, z1, zz

def energy_from_probs(p_z, p_x):
    z0, z1, zz = z_terms(p_z)
    xx = z_terms(p_x)[2]
    return coef["II"] + coef["IZ"] * z0 + coef["ZI"] * z1 + coef["ZZ"] * zz + coef["XX"] * xx

# Checkpoint: with exact probabilities the formula reproduces the Estimator
p_z_exact = probs_of(Statevector(bound).probabilities_dict())
x_basis = QuantumCircuit(2); x_basis.h([0, 1])
p_x_exact = probs_of(Statevector(bound).evolve(x_basis).probabilities_dict())
assert np.isclose(energy_from_probs(p_z_exact, p_x_exact), E_exact, atol=1e-9)

E_shots = energy_from_probs(probs_of(counts_of(z_circuit(bound))), probs_of(counts_of(x_circuit(bound))))
print(f"exact energy {E_exact:.4f}   from {SHOTS} shots per basis {E_shots:.4f}   difference {E_shots - E_exact:+.4f} hartree")
assert abs(E_shots - E_exact) < 0.05

### 6.2 Gate and readout error

Day 5's noisy simulator, built from the calibration data of the snapshot chosen in the cell below (`backend`), adds relaxation during every gate, depolarizing error at the calibrated rates, and readout error. A level-3 transpilation picks two good physical qubits, printed as `physical`; every circuit in this section is then pinned to those same two so that a readout calibration taken on them applies. The noisy runs use `NOISY_SHOTS` shots, set in the same cell, enough to put statistical noise well below the hardware bias.

In [ ]:
assert result is not None, "complete the VQE cell in section 4.2 first"
backend = FakeSherbrooke()
noisy_backend = AerSimulator.from_backend(backend, seed_simulator=17)     # the seed makes the local Estimator below reproducible
noise_rng = np.random.default_rng(5)

def next_seed():
    return int(noise_rng.integers(2 ** 31 - 1))

physical = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=1).run(z_circuit(bound)).layout.final_index_layout()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend, initial_layout=physical)
NOISY_SHOTS = 10_000

def run_noisy(qc, shots=NOISY_SHOTS):
    return noisy_backend.run(pm.run(qc), shots=shots, seed_simulator=next_seed()).result().get_counts()

print("physical qubits:", physical)
print("ISA circuit depth", pm.run(z_circuit(bound)).depth(), "with", pm.run(z_circuit(bound)).count_ops())

z_counts = run_noisy(z_circuit(bound))
x_counts = run_noisy(x_circuit(bound))
E_noisy = energy_from_probs(probs_of(z_counts), probs_of(x_counts))
print(f"noisy energy {E_noisy:.4f}   error {E_noisy - E_exact:+.4f} hartree")

### 6.3 Readout mitigation

The confusion matrix $M$ from Day 5 is measured the same way: prepare each of the four basis states on the same physical qubits, and record what is read back. Solving $M\vec p_{\rm true} = \vec p_{\rm meas}$ for each of the two measured distributions, then clipping and renormalizing, removes the readout part of the error before the expectation values are formed.

### Your turn: the mitigated energy

Complete `mitigated_energy(z_counts, x_counts, M)`. For each counts dictionary: turn it into a probability vector ordered as `labels`, solve `M @ p_true = p_meas` with `np.linalg.solve`, clip negatives to zero, renormalize, and rebuild a dictionary with `dict(zip(labels, p_true))`. Then return `energy_from_probs` of the two mitigated dictionaries.

In [ ]:
assert result is not None, "complete the VQE cell in section 4.2 first"
def calibration_matrix(shots=NOISY_SHOTS):
    M = np.zeros((4, 4))
    for j, label in enumerate(labels):
        qc = QuantumCircuit(2)
        for q, bit in enumerate(reversed(label)):
            if bit == "1":
                qc.x(q)
        qc.measure_all()
        counts = run_noisy(qc, shots)
        for i, out in enumerate(labels):
            M[i, j] = counts.get(out, 0) / shots
    return M

M = calibration_matrix()
print("confusion matrix diagonal:", np.diag(M))

def mitigated_energy(z_counts, x_counts, M):
    result = None
    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return result

In [ ]:
# Checkpoint: mitigation shrinks the error
assert result is not None, "complete the VQE cell in section 4.2 first"
E_mitigated = mitigated_energy(z_counts, x_counts, M)
assert E_mitigated is not None, "mitigated_energy returns None: fill in the cell above"
assert abs(E_mitigated - E_exact) < abs(E_noisy - E_exact), (E_mitigated, E_noisy)

print(f"{'energy at the optimal angles':<34}{'hartree':>10}{'error':>10}")
for name, e in [("exact", E_exact), (f"{SHOTS} shots, no other noise", E_shots), ("noisy simulator", E_noisy), ("noisy, readout mitigated", E_mitigated)]:
    print(f"{name:<34}{e:>10.4f}{e - E_exact:>+10.4f}")

What mitigation leaves behind happened inside the circuit: the two-qubit gate (the snapshot's `ecr`) and the relaxation during it. Day 5's zero-noise extrapolation attacks that part. The runtime Estimator does the readout calibration itself when `resilience_level = 1` is set (a method called TREX); in local testing mode that option is ignored, which is why this section did it by hand. The local-mode call below is the same call the live-hardware section makes, with resilience left off, and it should land near `E_noisy`.

In [ ]:
assert result is not None, "complete the VQE cell in section 4.2 first"
isa = pm.run(bound)
local_estimator = Estimator(mode=noisy_backend)
local_estimator.options.default_shots = NOISY_SHOTS
E_local = float(local_estimator.run([(isa, H_mol.apply_layout(isa.layout))]).result()[0].data.evs)
print(f"runtime Estimator in local testing mode: {E_local:.4f}   error {E_local - E_exact:+.4f} hartree")

## 7. Ansatz design and the dissociation curve

### 7.1 Two ansätze for the same molecule

The four-parameter hardware-efficient ansatz found the ground state, but so did the one-parameter circuit of section 4.1, and it is not luck. $H_{\rm mol}$ couples $|01\rangle$ only to $|10\rangle$ (through $XX$), and the ground state is the combination $\cos\frac\theta2|01\rangle + \sin\frac\theta2|10\rangle$ that lowers the energy most. That is the two-qubit form of the **unitary coupled cluster** ansatz, $e^{-i\theta X_0Y_1}|01\rangle$, which O'Malley et al. used: it is built from what the Hamiltonian conserves, starts at the Hartree-Fock state when $\theta = 0$, and has nothing to learn that the physics already rules out. A **problem-inspired** ansatz like this is cheap to optimize; a **hardware-efficient** ansatz is generic, covers states the problem never needs, and its gradients shrink with qubit count (Day 7 measures that). Both from the same start:

In [ ]:
assert result is not None, "complete the VQE cell in section 4.2 first"
def counted(f):
    def g(p):
        evaluations[0] += 1
        return f(p)
    return g

evaluations[0] = 0
r_he = minimize(counted(energy), x0, method="COBYLA", options={"maxiter": 300})
n_he = evaluations[0]

evaluations[0] = 0
r_one = minimize(counted(lambda p: float(estimator.run([(one_parameter(p[0]), H_mol)]).result()[0].data.evs)), [x0[0]], method="COBYLA", options={"maxiter": 300})
n_one = evaluations[0]

print(f"hardware efficient, 4 parameters: error {r_he.fun - E0:.1e} hartree after {n_he} evaluations")
print(f"one parameter:                    error {r_one.fun - E0:.1e} hartree after {n_one} evaluations")

# Checkpoint: both reach E0; the problem-inspired ansatz needs fewer evaluations
assert r_he.fun - E0 < 1e-3 and r_one.fun - E0 < 1e-3
assert n_one < n_he

### 7.2 The dissociation curve of $\mathrm{H}_2$

A single energy is not what chemistry wants; the energy as a function of bond length $R$ is. O'Malley et al. (2016) tabulated the two-qubit Hamiltonian at 54 bond lengths,

$$H(R) = g_0\,I + g_1\,Z_0 + g_2\,Z_1 + g_3\,Z_0Z_1 + g_4\,X_0X_1 + g_4\,Y_0Y_1 ,$$

in the STO-6G basis with the Bravyi-Kitaev mapping, and with the nuclear repulsion already inside $g_0$. The rows from 0.30 to 2.50 Å are below (the $Y_0Y_1$ coefficient equals the $X_0X_1$ one at every $R$). Because the basis differs from the STO-3G one behind $H_{\rm mol}$, its equilibrium energy differs too, $-1.1456$ against $-1.1373$ hartree: the number depends on the basis set, and only the shape of the curve is comparable between them.

In [ ]:
# R (angstrom), g0, g1, g2, g3, g4 from O'Malley et al., Phys. Rev. X 6, 031007 (2016), Table I
h2_table = np.array([
    [0.30, 1.7252, 0.5215, -1.1458, 0.6631, 0.0806], [0.35, 1.3827, 0.4982, -1.0226, 0.6537, 0.0815],
    [0.40, 1.1182, 0.4754, -0.9145, 0.6438, 0.0825], [0.45, 0.9083, 0.4534, -0.8194, 0.6336, 0.0835],
    [0.50, 0.7381, 0.4325, -0.7355, 0.6233, 0.0846], [0.55, 0.5979, 0.4125, -0.6612, 0.6129, 0.0858],
    [0.60, 0.4808, 0.3937, -0.5950, 0.6025, 0.0870], [0.65, 0.3819, 0.3760, -0.5358, 0.5921, 0.0883],
    [0.70, 0.2976, 0.3593, -0.4826, 0.5818, 0.0896], [0.75, 0.2252, 0.3435, -0.4347, 0.5716, 0.0910],
    [0.80, 0.1626, 0.3288, -0.3915, 0.5616, 0.0925], [0.85, 0.1083, 0.3149, -0.3523, 0.5518, 0.0939],
    [0.90, 0.0609, 0.3018, -0.3168, 0.5421, 0.0954], [0.95, 0.0193, 0.2895, -0.2845, 0.5327, 0.0970],
    [1.00, -0.0172, 0.2779, -0.2550, 0.5235, 0.0986], [1.05, -0.0493, 0.2669, -0.2282, 0.5146, 0.1002],
    [1.10, -0.0778, 0.2565, -0.2036, 0.5059, 0.1018], [1.15, -0.1029, 0.2467, -0.1810, 0.4974, 0.1034],
    [1.20, -0.1253, 0.2374, -0.1603, 0.4892, 0.1050], [1.25, -0.1452, 0.2286, -0.1413, 0.4812, 0.1067],
    [1.30, -0.1629, 0.2203, -0.1238, 0.4735, 0.1083], [1.35, -0.1786, 0.2123, -0.1077, 0.4660, 0.1100],
    [1.40, -0.1927, 0.2048, -0.0929, 0.4588, 0.1116], [1.45, -0.2053, 0.1976, -0.0792, 0.4518, 0.1133],
    [1.50, -0.2165, 0.1908, -0.0666, 0.4451, 0.1149], [1.55, -0.2265, 0.1843, -0.0549, 0.4386, 0.1165],
    [1.60, -0.2355, 0.1782, -0.0442, 0.4323, 0.1181], [1.65, -0.2436, 0.1723, -0.0342, 0.4262, 0.1196],
    [1.70, -0.2508, 0.1667, -0.0251, 0.4204, 0.1211], [1.75, -0.2573, 0.1615, -0.0166, 0.4148, 0.1226],
    [1.80, -0.2632, 0.1565, -0.0088, 0.4094, 0.1241], [1.85, -0.2684, 0.1517, -0.0015, 0.4042, 0.1256],
    [1.90, -0.2731, 0.1472, 0.0052, 0.3992, 0.1270], [1.95, -0.2774, 0.1430, 0.0114, 0.3944, 0.1284],
    [2.00, -0.2812, 0.1390, 0.0171, 0.3898, 0.1297], [2.05, -0.2847, 0.1352, 0.0223, 0.3853, 0.1310],
    [2.10, -0.2879, 0.1316, 0.0272, 0.3811, 0.1323], [2.15, -0.2908, 0.1282, 0.0317, 0.3769, 0.1335],
    [2.20, -0.2934, 0.1251, 0.0359, 0.3730, 0.1347], [2.25, -0.2958, 0.1221, 0.0397, 0.3692, 0.1359],
    [2.30, -0.2980, 0.1193, 0.0432, 0.3655, 0.1370], [2.35, -0.3000, 0.1167, 0.0465, 0.3620, 0.1381],
    [2.40, -0.3018, 0.1142, 0.0495, 0.3586, 0.1392], [2.45, -0.3035, 0.1119, 0.0523, 0.3553, 0.1402],
    [2.50, -0.3051, 0.1098, 0.0549, 0.3521, 0.1412],
])
bond_lengths = h2_table[:, 0]

def h2_hamiltonian(R):
    g0, g1, g2, g3, g4 = h2_table[np.isclose(bond_lengths, R)][0, 1:]
    return SparsePauliOp.from_list([("II", g0), ("IZ", g1), ("ZI", g2), ("ZZ", g3), ("XX", g4), ("YY", g4)])

exact_curve = np.array([np.linalg.eigvalsh(h2_hamiltonian(R).to_matrix()).min() for R in bond_lengths])
hf_curve = np.array([Statevector.from_label("01").expectation_value(h2_hamiltonian(R)).real for R in bond_lengths])
print(f"exact energy at 0.75 A: {exact_curve[np.isclose(bond_lengths, 0.75)][0]:.4f} hartree")

### Your turn: VQE at every bond length

Complete `energy_at(R)`: build `h2_hamiltonian(R)`, minimize the energy of `one_parameter(t)` over the angle `t` with `scipy.optimize.minimize_scalar(..., bounds=(-np.pi, np.pi), method="bounded")`, and return the minimum energy as a float.

In [ ]:
from scipy.optimize import minimize_scalar

def energy_at(R):
    result = None

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return result

In [ ]:
# Checkpoint: VQE follows the exact curve, and the minimum sits at 0.75 A
e_test = energy_at(0.75)
assert e_test is not None, "energy_at returns None: fill in the cell above"
vqe_curve = np.array([energy_at(R) for R in bond_lengths])
assert np.abs(vqe_curve - exact_curve).max() < 1e-4, np.abs(vqe_curve - exact_curve).max()
R_min = bond_lengths[np.argmin(vqe_curve)]
assert np.isclose(R_min, 0.75), R_min
D_e = vqe_curve[-1] - vqe_curve.min()
print(f"minimum {vqe_curve.min():.4f} hartree at R = {R_min:.2f} A;  E(2.5 A) - E(min) = {D_e:.4f} hartree = {D_e * 27.2114:.2f} eV")

plt.plot(bond_lengths, exact_curve, "-", color="0.4", label="exact eigenvalue")
plt.plot(bond_lengths, hf_curve, ":", label="Hartree-Fock state |01>")
plt.plot(bond_lengths, vqe_curve, "o", ms=4, label="VQE, one-parameter ansatz")
plt.xlabel("bond length R (angstrom)"); plt.ylabel("energy (hartree)"); plt.legend(frameon=False); plt.title("H$_2$ dissociation curve"); plt.show()

The Hartree-Fock state alone is off by about 0.02 hartree at equilibrium and by more than 0.2 hartree at 2.5 Å, where the molecule is coming apart and a single configuration cannot describe two separated atoms. The one angle of the ansatz captures that correlation at every $R$. The dissociation energy, the depth of the well, comes out near 5.5 eV in a minimal basis against the experimental 4.75 eV; the basis, not the algorithm, sets that error.

## 8. QAOA: a combinatorial problem as a Hamiltonian

### 8.1 MaxCut and QUBO

Take a graph and color each node 0 or 1. An edge is **cut** when its two nodes have different colors, and **MaxCut** asks for the coloring with the most cut edges. Small instances are easy by brute force; the general problem is NP-hard. Encode the coloring as a bitstring, one qubit per node, and the number of cut edges as an observable:

$$C = \sum_{(i,j)\in E} \frac{1 - Z_i Z_j}{2},$$

because $Z_iZ_j = +1$ when the two bits agree and $-1$ when they differ, so each term contributes 1 exactly for a cut edge. The bitstring with the largest $\langle C\rangle$ is the best cut. Any **QUBO** (quadratic unconstrained binary optimization, $\min x^\mathsf{T}Qx$ over bits $x$) becomes a Hamiltonian the same way, through $x_i = (1 - Z_i)/2$, which is what makes optimization problems a natural target for quantum algorithms.

In [ ]:
n = 4
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]         # a square: four nodes, four edges

def cut_value(bitstring):
    bits = bitstring[::-1]                        # qubit 0 is the rightmost character
    return sum(1 for i, j in edges if bits[i] != bits[j])

all_cuts = {format(k, f"0{n}b"): cut_value(format(k, f"0{n}b")) for k in range(2 ** n)}
best_cut = max(all_cuts.values())
print("cut of every coloring:", all_cuts)
print("best possible cut:", best_cut, "achieved by", [b for b, c in all_cuts.items() if c == best_cut])

### Your turn: the cost Hamiltonian

Complete `cost_hamiltonian(edges, n)` so that it returns $C$ as a `SparsePauliOp` on `n` qubits. `SparsePauliOp.from_sparse_list([("ZZ", [i, j], coeff), ...], num_qubits=n)` builds the $Z_iZ_j$ terms, and the constant $|E|/2$ is `SparsePauliOp("I" * n, coeff)`.

In [ ]:
def cost_hamiltonian(edges, n):
    C = None

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return C

In [ ]:
# Checkpoint: eigenvalues are the cut values, and the largest is the best cut
C = cost_hamiltonian(edges, n)
assert C is not None, "cost_hamiltonian returns None: fill in the cell above"
assert np.isclose(np.linalg.eigvalsh(C.to_matrix()).max(), best_cut)
for b in ("0000", "0101", "0011"):
    assert np.isclose(Statevector.from_label(b).expectation_value(C).real, all_cuts[b]), b
print("C =", C)

### 8.2 The QAOA circuit

The **quantum approximate optimization algorithm** (QAOA) uses an ansatz built from the problem itself. Start in $|+\rangle^{\otimes n}$, the uniform superposition of every coloring. Then alternate two layers $p$ times: a **cost layer** of one $R_{zz}(2\gamma)$ per edge, which is $e^{-i\gamma\sum Z_iZ_j}$ and gives each coloring a phase proportional to its cut (up to a global phase it equals $e^{-i\gamma' C}$ with $\gamma' = -2\gamma$), and a **mixer layer** $e^{-i\beta\sum X_i}$, one $R_x(2\beta)$ per qubit, which moves amplitude between colorings. Optimizing the $2p$ angles to maximize $\langle C\rangle$ concentrates the state on good cuts; then sampling it reads them out.

In [ ]:
def qaoa_circuit(edges, n, p):
    gammas = ParameterVector("gamma", p)
    betas = ParameterVector("beta", p)
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p):
        for i, j in edges:
            qc.rzz(2 * gammas[layer], i, j)
        qc.rx(2 * betas[layer], range(n))
        qc.barrier()
    return qc

p = 2
qaoa_qc = qaoa_circuit(edges, n, p)
qaoa_qc.draw("mpl", fold=-1)

In [ ]:
assert C is not None, "complete cost_hamiltonian above first"

def negative_cut(params):
    return -float(estimator.run([(qaoa_qc, C, params)]).result()[0].data.evs)

best = None
for restart in range(3):
    start = rng.uniform(0, np.pi, 2 * p)
    r = minimize(negative_cut, start, method="COBYLA", options={"maxiter": 300})
    print(f"restart {restart}: <C> = {-r.fun:.4f}")
    if best is None or r.fun < best.fun:
        best = r
print(f"\nbest expected cut {-best.fun:.4f} of {best_cut}   angles {best.x}")

### Your turn: read out the answer

Bind `best.x` to `qaoa_qc`, add measurements, sample it with `counts_of`, and store the counts in `qaoa_counts`. The most frequent bitstrings should be maximum cuts.

In [ ]:
qaoa_counts = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint: the most likely outcome is a maximum cut
assert qaoa_counts is not None, "fill in the cell above"
top = max(qaoa_counts, key=qaoa_counts.get)
assert cut_value(top) == best_cut, f"top outcome {top} has cut {cut_value(top)}"
good = sum(v for k, v in qaoa_counts.items() if cut_value(k) == best_cut) / SHOTS
print(f"most frequent: {top} (cut {cut_value(top)});  {good:.0%} of samples are maximum cuts")
plot_histogram(qaoa_counts, title=f"QAOA, p = {p}: samples of the optimized state")

On this four-node square, $p = 2$ should put nearly all the probability on the two perfect cuts, `0101` and `1010`; the histogram above shows what your run found. On larger graphs the approximation ratio grows with $p$, each layer costing another round of two-qubit gates, and whether QAOA can beat the best classical heuristics at useful sizes is an open question.

## 9. An energy on the live machine (optional)

The VQE energy at the optimal angles is evaluated on a real processor with the runtime Estimator and its built-in readout mitigation, about 3 seconds of QPU time. Three numbers to compare: the exact eigenvalue (a classical calculation), the noisy simulator's mitigated energy from section 6.3, and the processor. The gap between the last two is whatever the calibration snapshot did not predict.

In [ ]:
assert "E_mitigated" in globals(), "complete section 6.3 first"

job = None
if RUN_ON_HARDWARE and service is not None:
    live = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
    pm_live = generate_preset_pass_manager(optimization_level=3, backend=live)
    isa = pm_live.run(bound)
    H_isa = H_mol.apply_layout(isa.layout)          # move the observable onto the physical qubits
    est = Estimator(mode=live)
    est.options.resilience_level = 1
    job = est.run([(isa, H_isa)])
    print(f"Submitted to {live.name}, job id {job.job_id()}")
    e_hw = float(job.result()[0].data.evs)
    for name, e in [("exact eigenvalue", E0), ("noisy simulator, mitigated", E_mitigated), (f"{live.name}, resilience 1", e_hw)]:
        print(f"{name:<32} {e:.4f}   error {e - E0:+.4f} hartree")
else:
    # Recorded run: ibm_marrakesh (Heron r2), physical qubits 14 and 15, ISA depth 11 with one cz, runtime Estimator with resilience_level = 1, 4096 shots, job dam7batr85ps73fcbtb0
    CACHED_HW_ENERGY, CACHED_HW_STD = -1.8471, 0.0041
    print("RUN_ON_HARDWARE is False or no account: using the recorded hardware energy below.")
    for name, e in [("exact eigenvalue", E0), ("noisy simulator, mitigated", E_mitigated), ("recorded run, resilience 1", CACHED_HW_ENERGY)]:
        print(f"{name:<32} {e:.4f}   error {e - E0:+.4f} hartree")
    print(f"(the recorded run's statistical error bar is {CACHED_HW_STD:.4f} hartree)")

## 10. Summary

- A parameterized circuit is a circuit with symbolic angles; an ansatz is such a circuit used as a search space. The hardware-efficient ansatz has $n(\text{reps}+1)$ parameters.
- A Hamiltonian is an observable written in Pauli strings. The variational principle, $\langle\psi|H|\psi\rangle \ge E_0$, makes the lowest energy over an ansatz an upper bound on the ground-state energy.
- VQE minimizes the Estimator's energy with a classical optimizer. For two-qubit $\mathrm{H}_2$ it reaches the electronic energy $E_0 = -1.8573$ hartree to high precision.
- The parameter-shift rule gives exact gradients from two energy evaluations per parameter. Gradient descent, SPSA, and COBYLA all reach $E_0$ on the exact energy; SPSA's two evaluations per step make it the usual choice under noise.
- On a processor the energy is assembled from counts in two measurement bases. Shot noise, gate error, and readout error each shift it; readout mitigation with the confusion matrix removes the last part.
- A problem-inspired ansatz reaches the ground state with fewer parameters and fewer evaluations than a hardware-efficient one. VQE at 45 bond lengths reproduces the $\mathrm{H}_2$ dissociation curve, with its minimum at 0.75 Å in the STO-6G basis.
- MaxCut, and any QUBO, becomes a Hamiltonian through $Z_iZ_j$ terms. QAOA alternates cost and mixer layers and samples the optimized state; on the four-node square, $p = 2$ finds the perfect cuts.

## Further reading

- A. Peruzzo et al., "A variational eigenvalue solver on a photonic quantum processor," Nature Communications 5, 4213 (2014), the original VQE
- P. J. J. O'Malley et al., "Scalable quantum simulation of molecular energies," Phys. Rev. X 6, 031007 (2016), the two-qubit $\mathrm{H}_2$ experiment and its coefficient table (Table I)
- A. Kandala et al., "Hardware-efficient variational quantum eigensolver for small molecules and quantum magnets," Nature 549, 242 (2017)
- M. Schuld, V. Bergholm, C. Gogolin, J. Izaac, and N. Killoran, "Evaluating analytic gradients on quantum hardware," Phys. Rev. A 99, 032331 (2019), the parameter-shift rule
- J. C. Spall, "Implementation of the simultaneous perturbation algorithm for stochastic optimization," IEEE Transactions on Aerospace and Electronic Systems 34, 817 (1998)
- E. Farhi, J. Goldstone, and S. Gutmann, "A quantum approximate optimization algorithm," arXiv:1411.4028 (2014)
- [Variational algorithm design](https://quantum.cloud.ibm.com/learning/en/courses/variational-algorithm-design) course on IBM Quantum Learning